# Calimerge — `keypoints_3d.csv` viewer (CSV input)

Sister notebook to `test_output.ipynb`, but reads the long-format CSV that Calimerge writes alongside the `.npz` raw buffer.

**No imports from `calimerge` itself** — only `numpy`, `pandas`, `matplotlib`, and the standard library. The SynthPose-52 keypoint schema is inlined below, so this notebook keeps working even if the calimerge package is uninstalled / refactored / renamed in the future.

## Expected CSV header

```
time_s,sync_index,person_index,person_id,kp_index,x,y,z,valid[,camera_frame_time_s]
```

* `time_s` — detection-pipeline timestamp (seconds since recording start, set by the worker that buffered the keypoint).
* `sync_index` — index of the synchronized multi-camera frame.
* `person_index` — 0-based index within the frame's person list.
* `person_id` — stable track id (when available; falls back to `person_index`).
* `kp_index` — SynthPose-52 keypoint index (0–51).
* `x, y, z` — 3D position in metres. Empty when missing.
* `valid` — 1 if the (x, y, z) triple was finite when written, else 0.
* `camera_frame_time_s` *(optional, added by Task 1)* — wall-clock arrival time of the **first** camera's frame for this `sync_index`, in seconds since recording start. Lets the student compare detection-side vs camera-side timing and see the (small) processing offset.

## What this notebook plots

1. Skeleton at start / middle / end frames (3D, blue = left, red = right)
2. Foot (left + right ankle) position over time
3. Inter-frame `dt` diagnostic (frame skip detection)
4. `time_s` vs `camera_frame_time_s` comparison panel (when the column is present)

## 1. Pick the CSV via a file browser

In [ ]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog

def pick_csv(default_dir: str | None = None) -> Path:
    """Pop a native file dialog and return the chosen .csv path."""
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    chosen = filedialog.askopenfilename(
        title="Pick keypoints_3d.csv",
        initialdir=default_dir or str(Path.home()),
        filetypes=[("CSV", "*.csv"), ("All files", "*.*")],
    )
    root.destroy()
    if not chosen:
        raise FileNotFoundError("No file selected.")
    return Path(chosen)

default_dir = str(Path.home() / "Documents" / "Calimerge")
if not Path(default_dir).is_dir():
    default_dir = None

csv_path = pick_csv(default_dir=default_dir)
print(f"Loaded: {csv_path}")

## 2. SynthPose-52 keypoint schema (inlined)

Calimerge writes 52 keypoints per person per frame in this fixed order. Indices 0–16 are the COCO-17 set; 17–51 are anatomical landmarks added by the SynthPose model. Side classification (`L` / `R` / `C`) is what makes left-vs-right plots possible.

In [ ]:
SYNTHPOSE_MARKERS = {
    0: "Nose", 1: "L_Eye", 2: "R_Eye", 3: "L_Ear", 4: "R_Ear",
    5: "L_Shoulder", 6: "R_Shoulder", 7: "L_Elbow", 8: "R_Elbow",
    9: "L_Wrist", 10: "R_Wrist", 11: "L_Hip", 12: "R_Hip",
    13: "L_Knee", 14: "R_Knee", 15: "L_Ankle", 16: "R_Ankle",
    17: "sternum", 18: "rshoulder", 19: "lshoulder", 20: "r_lelbow",
    21: "l_lelbow", 22: "r_melbow", 23: "l_melbow", 24: "r_lwrist",
    25: "l_lwrist", 26: "r_mwrist", 27: "l_mwrist", 28: "r_ASIS",
    29: "l_ASIS", 30: "r_PSIS", 31: "l_PSIS", 32: "r_knee",
    33: "l_knee", 34: "r_mknee", 35: "l_mknee", 36: "r_ankle",
    37: "l_ankle", 38: "r_mankle", 39: "l_mankle", 40: "r_5meta",
    41: "l_5meta", 42: "r_toe", 43: "l_toe", 44: "r_big_toe",
    45: "l_big_toe", 46: "l_calc", 47: "r_calc", 48: "C7",
    49: "L2", 50: "T11", 51: "T6",
}

def side_of(idx: int) -> str:
    """Anatomical side: 'L' / 'R' / 'C' (center)."""
    name = SYNTHPOSE_MARKERS.get(idx, "")
    if name.startswith(("L_", "l_")):
        return "L"
    if name.startswith(("R_", "r_")):
        return "R"
    return "C"

L_HIP, R_HIP = 11, 12
L_ANKLE, R_ANKLE = 15, 16

SKELETON_BONES = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
    (15, 41), (15, 43), (15, 46),
    (16, 40), (16, 42), (16, 47),
    (0, 48), (48, 51), (51, 50), (50, 49),
]

print(f"{len(SYNTHPOSE_MARKERS)} keypoints, {len(SKELETON_BONES)} bones drawn")

## 3. Load the CSV into a dense `(n_frames, max_persons, 52, 3)` array

We pivot the long-form CSV into the same shape `test_output.ipynb` uses, so plotting code is identical.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(csv_path)
print(f"loaded {len(df)} rows, columns: {list(df.columns)}")

n_kps = max(int(df["kp_index"].max()) + 1, 52)
n_frames = int(df["sync_index"].max()) + 1
max_persons = int(df["person_index"].max()) + 1 if len(df) else 1

kps = np.full((n_frames, max_persons, n_kps, 3), np.nan, dtype=np.float32)
times = np.zeros(n_frames, dtype=np.float64)
counts = np.zeros(n_frames, dtype=np.int32)
primary = np.zeros(n_frames, dtype=np.int32)

have_camera_frame_time = "camera_frame_time_s" in df.columns
camera_frame_time = (
    np.full(n_frames, np.nan, dtype=np.float64) if have_camera_frame_time else None
)

valid_rows = df[df["valid"] == 1]
for row in valid_rows.itertuples(index=False):
    fi = int(row.sync_index)
    pi = int(row.person_index)
    ki = int(row.kp_index)
    kps[fi, pi, ki] = (float(row.x), float(row.y), float(row.z))

for fi, group in df.groupby("sync_index"):
    times[int(fi)] = float(group["time_s"].iloc[0])
    counts[int(fi)] = int(group["person_index"].nunique())
    if have_camera_frame_time:
        cft = group["camera_frame_time_s"].dropna()
        if len(cft):
            camera_frame_time[int(fi)] = float(cft.iloc[0])

duration = float(times[-1] - times[0]) if n_frames > 1 else 0.0
fps = n_frames / duration if duration > 0 else float("nan")

print(f"frames={n_frames} | persons<= {max_persons} | keypoints={n_kps} | duration={duration:.2f} s | mean fps={fps:.1f}")
print(f"per-frame valid-person counts: min={counts.min()} max={counts.max()} mean={counts.mean():.2f}")
if have_camera_frame_time:
    cft_finite = camera_frame_time[~np.isnan(camera_frame_time)]
    print(f"camera_frame_time_s present on {len(cft_finite)}/{n_frames} frames")

## 4. Whole skeleton at a few frames

Three views (start / middle / end) of the primary person. Left-side keypoints draw blue, right-side red, midline gray.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the projection)

SIDE_COLOR = {"L": "#5099ff", "R": "#ff5050", "C": "#aaaaaa"}

def plot_skeleton(ax, frame_kps: np.ndarray, title: str = "") -> None:
    valid = ~np.isnan(frame_kps).any(axis=1)
    for i, j in SKELETON_BONES:
        if not (valid[i] and valid[j]):
            continue
        si, sj = side_of(i), side_of(j)
        if si == sj:
            color = SIDE_COLOR[si]
        elif "C" in (si, sj):
            color = SIDE_COLOR[si if si != "C" else sj]
        else:
            color = SIDE_COLOR["C"]
        xs = [frame_kps[i, 0], frame_kps[j, 0]]
        ys = [frame_kps[i, 1], frame_kps[j, 1]]
        zs = [frame_kps[i, 2], frame_kps[j, 2]]
        ax.plot(xs, ys, zs, color=color, linewidth=2)
    for k in range(len(frame_kps)):
        if not valid[k]:
            continue
        ax.scatter(
            frame_kps[k, 0], frame_kps[k, 1], frame_kps[k, 2],
            color=SIDE_COLOR[side_of(k)], s=18, depthshade=True,
        )
    ax.set_title(title)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_zlabel("z (m)")

valid_frames = [i for i in range(n_frames) if counts[i] > 0]
if not valid_frames:
    raise RuntimeError("No frames contain any valid person — nothing to plot.")

picks = [
    valid_frames[0],
    valid_frames[len(valid_frames) // 2],
    valid_frames[-1],
]

fig = plt.figure(figsize=(15, 5))
for col, fi in enumerate(picks):
    pi = int(primary[fi]) if primary[fi] < counts[fi] else 0
    ax = fig.add_subplot(1, 3, col + 1, projection="3d")
    plot_skeleton(ax, kps[fi, pi], title=f"frame {fi}  t={times[fi]:.2f} s")

fig.suptitle("Primary person — start / middle / end")
fig.tight_layout()
plt.show()

## 5. Foot position over time

Plots ankle x / y / z for left + right foot of the primary person. The vertical (z) trace makes step cycles obvious.

In [ ]:
left_ankle = np.full((n_frames, 3), np.nan, dtype=np.float32)
right_ankle = np.full((n_frames, 3), np.nan, dtype=np.float32)
for fi in range(n_frames):
    if counts[fi] == 0:
        continue
    pi = int(primary[fi]) if primary[fi] < counts[fi] else 0
    left_ankle[fi] = kps[fi, pi, L_ANKLE]
    right_ankle[fi] = kps[fi, pi, R_ANKLE]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, axis_idx, axis_label in zip(axes, range(3), ("x (m)", "y (m)", "z (m)")):
    ax.plot(times, left_ankle[:, axis_idx], color=SIDE_COLOR["L"], label="L_Ankle")
    ax.plot(times, right_ankle[:, axis_idx], color=SIDE_COLOR["R"], label="R_Ankle")
    ax.set_ylabel(axis_label)
    ax.grid(True, alpha=0.3)
axes[0].set_title("Ankle position over time (primary person)")
axes[-1].set_xlabel("time (s)")
axes[0].legend(loc="upper right")
fig.tight_layout()
plt.show()

## 6. Inter-frame dt — frame-skip diagnostic

Spikes ≥ 2× the median usually mean either the cameras choked or the detection worker stalled.

In [ ]:
dts = np.diff(times)
median_dt = float(np.median(dts)) if len(dts) else float("nan")
skip_threshold = 2.0 * median_dt
skips = np.where(dts > skip_threshold)[0]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(times[1:], dts * 1000.0, color="#444", linewidth=1, label="dt (ms)")
ax.axhline(median_dt * 1000.0, color="#4caf50", linestyle="--",
           label=f"median {median_dt * 1000:.1f} ms")
ax.axhline(skip_threshold * 1000.0, color="#ff5050", linestyle=":",
           label=f"skip threshold {skip_threshold * 1000:.1f} ms")
for s in skips:
    ax.axvspan(times[s], times[s + 1], color="#ff5050", alpha=0.15)
ax.set_xlabel("time (s)")
ax.set_ylabel("frame interval (ms)")
ax.set_title(f"Inter-frame dt — {len(skips)} apparent skip(s) of >{skip_threshold * 1000:.0f} ms")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print(f"frames: {n_frames}, total duration: {duration:.2f} s")
if median_dt:
    print(f"median dt: {median_dt * 1000:.2f} ms  -> ~{1.0 / median_dt:.1f} fps")
if len(dts):
    print(f"max dt:    {dts.max() * 1000:.2f} ms")
print(f"#skips (dt > 2x median): {len(skips)}")

## 7. `time_s` vs `camera_frame_time_s`

When the CSV includes the `camera_frame_time_s` column (Task 1), this panel overlays both timestamps so the student can see the (typically small) offset between the detection-side `time_s` and the camera's actual frame-arrival wall-clock. The two should track each other almost exactly; a growing gap means detection is falling behind capture.

In [ ]:
if not have_camera_frame_time:
    print("camera_frame_time_s column not present in this CSV — nothing to compare.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    axes[0].plot(times, camera_frame_time, label="camera_frame_time_s", color="#4caf50")
    axes[0].plot(times, times, label="time_s", color="#444", linestyle="--")
    axes[0].set_ylabel("seconds since\nrecording start")
    axes[0].set_title("Detection-side time_s vs camera-side camera_frame_time_s")
    axes[0].legend(loc="upper left")
    axes[0].grid(True, alpha=0.3)

    delta = camera_frame_time - times
    axes[1].plot(times, delta * 1000.0, color="#ff7043")
    axes[1].axhline(0.0, color="#888", linewidth=0.8)
    axes[1].set_xlabel("time_s (s)")
    axes[1].set_ylabel("camera - detect (ms)")
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

    finite_delta = delta[~np.isnan(delta)]
    if len(finite_delta):
        print(f"mean (camera - detect): {finite_delta.mean() * 1000:.2f} ms")
        print(f"max  (camera - detect): {finite_delta.max() * 1000:.2f} ms")